# Qwen-Image-2.1 图片编辑队列 V5.1｜单图 + 批量 ZIP

**ChatGPT 对话接单 → 审核/优化编辑提示词 → ZIP 批量入队 → Colab 免费 GPU 运行 → 自动上传 GitHub PNG → ChatGPT 定时通知。**

**V5.1 修复：Qwen-2.1 新架构兼容 GGUF 加载器、自动重启 ComfyUI、GitHub 写入预检、错误可重试。**

使用说明：[BATCH_PROTOCOL_V5.md](https://github.com/mjw3177109/qwen2.1tu/blob/main/BATCH_PROTOCOL_V5.md)。仓库当前公开：母图/成品均可被看到，不能用于隐私图片。免费 Colab GPU 无法在聊天中远程自动唤醒，也不应作为后台常驻 Worker。

**运行方式**：
1. 配置 GPU，运行第 01 格检查凭据/磁盘。
2. 如果本批母图与提示词**尚未入队**，运行第 02 格上传经过整理的 ZIP，一次完成所有任务的提交；如果 ChatGPT 已把母图和 pending JSON 真实上传到仓库，可跳过此格。
3. 顺序运行安装、模型下载、ComfyUI、工作流、执行待办格。生成结果保存在仓库 `images/<job_id>.png`。
4. 每次新任务到达后，Colab 在线时再运行「执行待办」格。这里不是实时后台 GPU 服务。

**提示词优化边界**：本 Notebook 负责机械配对、约束拼接、哈希检查和队列执行；语义层面的专业提示词优化由 ChatGPT 在提交前完成。

In [ ]:
# 01 / 配置 + GPU、磁盘 + GitHub Token 预检查（先执行，避免白下载模型）
from pathlib import Path
import os, sys, shutil, subprocess, time, io, json, re, uuid, base64
import requests

REPO = "mjw3177109/qwen2.1tu"
BRANCH = "main"
EDIT_PROMPT = "<image1> 保持人物身份、姿态、背景与二维动漫风格不变，只把衣服变成深蓝色。"
EDIT_RESOLUTION = 512       # 先用 512；成功后可尝试 640 或 768
ENCODER_MODE = "w4a8"      # 小体积 6.31GB；不兼容可试 int8（更占磁盘）
SEED = 42
ROOT = Path("/content/qwen21_light")
COMFY = ROOT / "ComfyUI"
OUT = Path("/content/qwen21_outputs")
OUT.mkdir(parents=True, exist_ok=True)
(ROOT / "tmp").mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(ROOT / "hf_cache")
os.environ["HF_HUB_CACHE"] = str(ROOT / "hf_cache/hub")
os.environ["HF_XET_CACHE"] = str(ROOT / "hf_cache/xet")
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
os.environ["PIP_NO_CACHE_DIR"] = "1"
os.environ["TMPDIR"] = str(ROOT / "tmp")
os.environ["XDG_CACHE_HOME"] = str(ROOT / "xdg_cache")

from google.colab import userdata
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise RuntimeError("Colab Secrets 中没有 GITHUB_TOKEN，或者当前 Notebook 未被授权访问。")
# GitHub PAT 不允许首尾空格；不要误加 Bearer 前缀。
GITHUB_TOKEN = GITHUB_TOKEN.strip()
if GITHUB_TOKEN.startswith(("Bearer ", "token ")):
    raise RuntimeError("GITHUB_TOKEN 请只填 GitHub 生成的原始 Token，不要包含 Bearer/token 前缀。")
if not GITHUB_TOKEN or any(c.isspace() for c in GITHUB_TOKEN):
    raise RuntimeError("GITHUB_TOKEN 含有空格、换行或为空。请在 Colab Secrets 中重新粘贴完整 Token。")

GH_HEADERS = {
    "Authorization": "Bearer " + GITHUB_TOKEN,
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
# 先验证 Token 本身，再验证指定仓库的访问权限。
auth_check = requests.get("https://api.github.com/user", headers=GH_HEADERS, timeout=25)
if auth_check.status_code == 401:
    raise RuntimeError(
        "⛔ GitHub HTTP 401：Colab 当前 GITHUB_TOKEN 无效、已撤销或未正确保存。"
        "请到 Colab 左侧 Secrets 更新 Value，开启 Notebook access，然后重新运行当前配置单元。"
        "不要将 Token 发到聊天中，也不要重新下载模型。")
if not auth_check.ok:
    raise RuntimeError(f"GitHub 身份验证预检失败 HTTP {auth_check.status_code}。"
                       "请核对 PAT 状态、账户和网络连接。")
print("✅ GitHub Token 已通过身份验证，账户：", auth_check.json().get("login","(未知)"))
probe = requests.get("https://api.github.com/repos/" + REPO, headers=GH_HEADERS, timeout=25)
if probe.status_code == 404:
    raise RuntimeError("⛔ GitHub 仓库不可访问。检查 Fine-grained PAT 是否授权"
                       "仓库 mjw3177109/qwen2.1tu。")
if probe.status_code == 401:
    raise RuntimeError("⛔ GitHub HTTP 401：仓库校验时 Token 被拒绝。请重新读取 Colab Secret。")
if probe.status_code != 200:
    raise RuntimeError("GitHub 仓库访问预检 HTTP " + str(probe.status_code)
                       + "。请检查 Token 的仓库权限。")
print("GitHub 仓库连接成功:", REPO)
# 真实写权限预检：GET 可能成功而 GitHub Contents PUT 被拒绝。
# 在大模型下载之前先发现此类问题，避免浪费 GPU 配额。
from datetime import datetime, timezone
preflight_path = "queue/runner_probe.json"
preflight_url = f"https://api.github.com/repos/{REPO}/contents/{preflight_path}"
current = requests.get(preflight_url,headers=GH_HEADERS,params={"ref":BRANCH},timeout=25)
if current.status_code not in (200,404):
    raise RuntimeError(f"GitHub 写入预检读取失败 HTTP {current.status_code}")
probe_obj = {"runner":"colab-qwen-edit-v5","checked_at":
    datetime.now(timezone.utc).isoformat(),"status":"CAN_WRITE"}
body = {"message":"test: verify Colab GitHub Contents write access",
        "content":base64.b64encode(json.dumps(probe_obj).encode("utf-8")).decode("ascii"),
        "branch":BRANCH}
if current.status_code==200:
    body["sha"]=current.json()["sha"]
w = requests.put(preflight_url,headers=GH_HEADERS,json=body,timeout=35)
if w.status_code not in (200,201):
    raise RuntimeError(
        f"⛔ Colab 的 GITHUB_TOKEN 只有读取或缺少写入权限。"
        f"请在 GitHub Fine-grained Token 对仓库 qwen2.1tu 启用 Contents: Read and write。"
        f"HTTP {w.status_code} / {w.text[:250]}")
r = requests.get(preflight_url,headers=GH_HEADERS,params={"ref":BRANCH},timeout=25)
if not r.ok or r.json().get("sha")!=w.json().get("content",{}).get("sha"):
    raise RuntimeError("GitHub 写权限预检：写入成功但回读校验失败")
print("✅ Colab GITHUB_TOKEN 实测可写入、可回读 GitHub；后续结果上传权限正常。")

if not probe.json().get("permissions", {}).get("push", False):
    print("⚠️ Token 未被确认具有 push 权限；请确保仓库 Contents 是 Read and write。")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("未检测到 GPU，请在运行时设置里选择 GPU。Colab 免费额度可能暂不可用。")
for i in range(torch.cuda.device_count()):
    prop = torch.cuda.get_device_properties(i)
    print("GPU:", prop.name, f"{prop.total_memory/2**30:.1f} GiB")
free = shutil.disk_usage("/content").free
print("Colab 临时磁盘剩余:", round(free/2**30, 1), "GiB")
if free < 17 * 2**30:
    raise RuntimeError("临时磁盘不足 17 GiB，停止以避免下载失败。")
try:
    import psutil
    print("系统总 RAM:", round(psutil.virtual_memory().total/2**30, 1), "GiB")
except ImportError:
    pass
assert ENCODER_MODE in ("w4a8", "int8")

## 第 02 格：批量入队（先检查再提交）

上传之前，可以把图片、txt、CSV 打包给 ChatGPT，我会帮你检查母图和提示词配对并优化成适用于 Qwen2.1 的版本。如果原始图片尚未上传到 GitHub，就在这里一次性上传 ZIP。没有待入队图片时，跳过这一格。

会先完整检查 ZIP，发现配对冲突就报错，不提交半成品；验证通过才先写 `jobs/inputs/` 再写 `jobs/pending/`。

In [ ]:
# 02 / 批量 ZIP 文件导入并提交 GitHub 队列
# 适用：单图 ZIP、多图同一修改、多图独立 txt、一图多改 batch.csv
# 注意：这里验证 + 入队，不生成任何图片。
from google.colab import files as colab_files
import importlib.util, base64, requests
from pathlib import Path
import json, sys

# 正常接单：ChatGPT 已将母图/提示词写入 GitHub，默认无需重复上传 ZIP。
# 如需要直接在 Colab 手工上传新的批量 ZIP，请改成 True。
IMPORT_NEW_ZIP = False
if not IMPORT_NEW_ZIP:
    print("✅ 使用 GitHub 已入队的任务，跳过 ZIP 上传。直接继续安装与 GPU 执行。")
else:
    print("请上传已整理/已优化的批量编辑 ZIP（整个 ZIP 一次上传）")
    uploaded_zips = colab_files.upload()
    if len(uploaded_zips) != 1:
        raise ValueError("一次只能上传一个 ZIP")
    zip_name, zip_bytes = next(iter(uploaded_zips.items()))
    if not zip_name.lower().endswith(".zip"):
        raise ValueError("文件必须为 ZIP")
    zip_path = Path("/content/qwen_edit_batch.zip")
    zip_path.write_bytes(zip_bytes)


    source_url = f"https://api.github.com/repos/{REPO}/contents/tools/qwen_batch_ingest.py"
    res = requests.get(source_url, headers=GH_HEADERS, params={"ref":BRANCH}, timeout=45)
    res.raise_for_status()
    response = res.json()
    if response.get("encoding") != "base64":
        raise RuntimeError("Github 返回的 batch module 编码不支持")
    code = base64.b64decode(response["content"])
    script_path = Path("/content/qwen_batch_ingest.py")
    script_path.write_bytes(code)
    spec = importlib.util.spec_from_file_location("qwen_batch_ingest",script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    
    plan = module.prepare_batch(zip_path)
    print(module.preview_as_text(plan))
    print("已通过图片数量、配对、字段、哈希、路径、任务 ID 等校验。")
    print("注意：仓库是公开的，上传的母图也会被公开访问。")
    report = module.submit_batch(plan, repo=REPO, token=GITHUB_TOKEN, branch=BRANCH)
    print("✅ 入队成功：",report["submitted"],"个任务")
    print("生成图片需要完成后续的 ComfyUI 模型加载并运行队列执行单元。")


In [ ]:
# 02 / 安装 ComfyUI + GGUF 扩展（复用 Colab 自带 torch）
def run_cmd(args, cwd=None):
    subprocess.run(list(map(str, args)), cwd=str(cwd) if cwd else None, check=True)

if not (COMFY / "main.py").exists():
    run_cmd(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", COMFY])
# Qwen-Image-2.1 必须使用支持 ModelQwenImage 的 leejet fork。
# 老 city96 版即使节点能出现，也会在真正加载 GGUF 时抛 Unknown model architecture!
node_dir = COMFY / "custom_nodes" / "ComfyUI-GGUF"
GGUF_REPO = "https://github.com/leejet/ComfyUI-GGUF.git"
legacy_dir = ROOT / "backups" / "ComfyUI-GGUF-city96"
legacy_dir.parent.mkdir(parents=True,exist_ok=True)

def git_remote(path):
    if not (path/".git").exists():
        return ""
    proc=subprocess.run(["git","-C",str(path),"remote","get-url","origin"],
                        capture_output=True,text=True)
    return proc.stdout.strip() if proc.returncode==0 else ""

if node_dir.exists() and "leejet/ComfyUI-GGUF" not in git_remote(node_dir):
    if legacy_dir.exists():
        shutil.rmtree(legacy_dir)
    shutil.move(str(node_dir),str(legacy_dir))
    print("旧版 GGUF 加载器已移出 custom_nodes 并备份:",legacy_dir)

if not (node_dir/"nodes.py").is_file():
    if node_dir.exists():
        raise RuntimeError("GGUF 节点目录不完整，请检查：" + str(node_dir))
    run_cmd(["git","clone","--depth","1",GGUF_REPO,node_dir])
else:
    # 新建的 Colab 会话无需额外下载；同一会话升级时刷新 fork。
    run_cmd(["git","-C",str(node_dir),"pull","--ff-only"])

git_rev=subprocess.check_output(["git","-C",str(node_dir),"rev-parse","--short","HEAD"],text=True).strip()
print("✅ Qwen-Image-2.1 GGUF 加载器: leejet/ComfyUI-GGUF @",git_rev)

requirements = []
for line in (COMFY / "requirements.txt").read_text(encoding="utf-8").splitlines():
    s = line.strip()
    if s and not s.startswith("#") and not re.match(r"^(torch|torchvision|torchaudio)(\b|[<>=~!\[])", s, flags=re.I):
        requirements.append(line)
req = ROOT / "requirements_without_torch.txt"
req.write_text("\n".join(requirements) + "\n", encoding="utf-8")
run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "-r", req])
run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "huggingface_hub>=0.33", "gguf", "requests"])
gguf_req = node_dir / "requirements.txt"
if gguf_req.exists():
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "-r", gguf_req])
print("ComfyUI 环境安装完成。")

In [ ]:
# 03 / 仅下载图片编辑需要的 3 个权重（模型存 Colab 临时磁盘，不保存到 GitHub）
from huggingface_hub import hf_hub_download

MODELS = COMFY / "models"
GGUF_FILE = "qwen-image-2.1-turbo-Q4_K_M.gguf"
ENC_FILE = ("qwen3vl_8b_w4a8.safetensors" if ENCODER_MODE == "w4a8"
            else "qwen3vl_8b_int8_convrot.safetensors")
VAE_FILE = "qwen_image_2.1_vae_bf16.safetensors"

weights = [
    ("abenzerps/Qwen-Image-2.1-Turbo-Quantized", GGUF_FILE, MODELS/"diffusion_models", 4.05),
    ("Comfy-Org/Qwen-Image-2.1", "text_encoders/"+ENC_FILE, MODELS, 6.31 if ENCODER_MODE=="w4a8" else 8.71),
    ("Comfy-Org/Qwen-Image-2.1", "vae/"+VAE_FILE, MODELS, 0.65),
]
for repo_id, filename, folder, size_gb in weights:
    folder.mkdir(parents=True, exist_ok=True)
    target = folder/filename
    if target.is_file() and target.stat().st_size > size_gb * 1e9 * 0.90:
        print("已存在，跳过:", target.name)
        continue
    if shutil.disk_usage("/content").free < size_gb*1e9 + 2e9:
        raise RuntimeError("磁盘不足，未下载 " + filename)
    print("下载:", repo_id, filename)
    hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(folder))
    if not target.is_file():
        raise FileNotFoundError("下载后找不到模型: " + str(target))
    print("完成:", target.name, round(target.stat().st_size/1e9, 2), "GB")
print("三个必需模型下载完成。")

In [ ]:
# 04 / 启动本地 ComfyUI 并检查所需节点
LOG = ROOT / "comfyui.log"
URL = "http://127.0.0.1:8188"
def healthy():
    try:
        return requests.get(URL+"/system_stats", timeout=3).ok
    except requests.RequestException:
        return False

# 重要：如果用户上次启动的是 city96 版，必须重启 ComfyUI；
# 仅升级节点目录，不重启 Python 服务，仍会引用旧的已加载模块。
if healthy():
    try:
        import psutil
    except ImportError:
        run_cmd([sys.executable,"-m","pip","install","-q","psutil"])
        import psutil
    stopped=False
    for process in psutil.process_iter(["pid","cmdline"]):
        try:
            args=process.info.get("cmdline") or []
            cmd=" ".join(args)
            if "main.py" in cmd and "--port" in cmd and "8188" in cmd and str(process.cwd())==str(COMFY):
                print("发现旧 ComfyUI 服务，重新加载 GGUF 插件：PID",process.pid)
                process.terminate()
                process.wait(timeout=15)
                stopped=True
        except (psutil.NoSuchProcess,psutil.AccessDenied,psutil.TimeoutExpired):
            continue
    time.sleep(2)
    if healthy():
        raise RuntimeError("8188 端口已有服务且无法安全终止。请重启 Colab 运行时，再执行所有单元格。")
    print("旧服务已停止，正在重新启动 ..." if stopped else "将启动全新 ComfyUI 服务。")

if not healthy():
    cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
           "--lowvram", "--disable-auto-launch", "--output-directory", str(OUT),
           "--temp-directory", str(ROOT/"tmp"), "--input-directory", str(COMFY/"input")]
    with LOG.open("a", encoding="utf-8") as f:
        comfy_process = subprocess.Popen(cmd, cwd=COMFY, stdout=f, stderr=subprocess.STDOUT)
    for _ in range(90):
        if healthy():
            break
        if comfy_process.poll() is not None:
            break
        time.sleep(2)
if not healthy():
    print("\n".join(LOG.read_text(errors="replace").splitlines()[-90:]))
    raise RuntimeError("ComfyUI 启动失败")

infos = requests.get(URL+"/object_info", timeout=45).json()
needed = ("UnetLoaderGGUF", "CLIPLoader", "VAELoader", "LoadImage",
          "TextEncodeQwenImage21", "KSamplerSelect", "ManualSigmas",
          "SamplerCustom", "VAEDecode", "SaveImage")
missing = [node for node in needed if node not in infos]
if missing:
    raise RuntimeError("当前 ComfyUI 版本缺少节点: " + ", ".join(missing))
print("编辑节点已加载，GGUF leejet @",git_rev,"，ComfyUI 已就绪。")

In [ ]:
# 06 / 图像编辑工作流（参考图确实接入 Qwen-Image-2.1 节点）
import secrets
SIGMAS = "1.0, 0.978453, 0.95418, 0.926626, 0.89508, 0.845148, 0.704534, 0.414568, 0.0"

def make_graph(prompt, input_name, seed):
    return {
        "1": {"class_type":"UnetLoaderGGUF","inputs":{"unet_name":GGUF_FILE}},
        "2": {"class_type":"CLIPLoader","inputs":{"clip_name":ENC_FILE, "type":"qwen_image", "device":"default"}},
        "3": {"class_type":"VAELoader","inputs":{"vae_name":VAE_FILE}},
        "4": {"class_type":"LoadImage","inputs":{"image":input_name}},
        "5": {"class_type":"TextEncodeQwenImage21","inputs":{
            "clip":["2",0], "vae":["3",0], "prompt":prompt, "negative_prompt":"",
            "resolution":EDIT_RESOLUTION, "images.image_1":["4",0]}},
        "6": {"class_type":"KSamplerSelect","inputs":{"sampler_name":"euler"}},
        "7": {"class_type":"ManualSigmas","inputs":{"sigmas":SIGMAS}},
        "8": {"class_type":"SamplerCustom","inputs":{
            "model":["1",0], "add_noise":True, "noise_seed":int(seed),
            "cfg":1.0, "positive":["5",0], "negative":["5",1],
            "sampler":["6",0], "sigmas":["7",0], "latent_image":["5",2]}},
        "9": {"class_type":"VAEDecode","inputs":{"samples":["8",0], "vae":["3",0]}},
        "10":{"class_type":"SaveImage","inputs":{
            "images":["9",0], "filename_prefix":"Qwen21_Edit_Github"}}
    }

# 校验参考图连接，防止误生成无关图片
assert make_graph("test", "example.png", SEED)["5"]["inputs"]["images.image_1"] == ["4",0]
print("图像编辑节点连接检查通过；生成任务按各自的母图路径和提示词加载。")

## 执行当前待办队列

前面 ComfyUI 节点检查完成后再运行下一格。处理最多 3 个待办任务，不常驻轮询。成功输出图片和结果 JSON；失败输出失败记录与通知事件。

In [ ]:
# 06 / 从 GitHub 待办目录一次性执行当前任务（非后台常驻 Worker）
# Colab 免费版请主动点击本单元；只处理当前队列，完成后结束。
import requests, io, json, base64, hashlib, uuid, time, re
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image, ImageOps
from IPython.display import display
GH_BASE = f"https://api.github.com/repos/{REPO}/contents"
MAX_JOBS_PER_RUN = 3
JOB_TIMEOUT_SECONDS = 2400
MAX_SOURCE_BYTES = 4_000_000

def now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00","Z")

def gh_read(path, missing_ok=False):
    r = requests.get(f"{GH_BASE}/{path}", headers=GH_HEADERS,
                     params={"ref":BRANCH}, timeout=40)
    if r.status_code == 404 and missing_ok:
        return None
    r.raise_for_status()
    return r.json()

def gh_read_text(path, missing_ok=False):
    data = gh_read(path, missing_ok=missing_ok)
    if data is None:
        return None, None
    if data.get("type") != "file" or data.get("encoding") != "base64":
        raise RuntimeError("不支持的 GitHub 文件响应：" + path)
    text_bytes = base64.b64decode(data["content"])
    return text_bytes.decode("utf-8-sig"), data.get("sha")

def gh_write_bytes(path, body: bytes, message: str, existing_sha=None):
    payload = {"message":message, "content":base64.b64encode(body).decode("ascii"),
               "branch":BRANCH}
    if existing_sha:
        payload["sha"] = existing_sha
    r = requests.put(f"{GH_BASE}/{path}", headers=GH_HEADERS, json=payload,timeout=120)
    if r.status_code not in (200,201):
        # Token 与请求头绝不输出
        raise RuntimeError(f"Github 提交失败 {path} HTTP {r.status_code}: {r.text[:350]}")
    expected_sha = r.json().get("content",{}).get("sha")
    remote = gh_read(path)
    if remote.get("sha") != expected_sha:
        raise RuntimeError("GitHub 文件写入后核验失败：" + path)
    return expected_sha

def gh_write_json(path, obj, existing_sha=None):
    body = (json.dumps(obj,ensure_ascii=False,indent=2)+"\n").encode("utf-8")
    return gh_write_bytes(path, body, f"jobs: record {path}", existing_sha)

def append_event(event):
    path = "queue/events.json"
    for attempt in range(5):
        body, sha = gh_read_text(path, missing_ok=True)
        data = json.loads(body) if body else {"schema_version":1,"events":[]}
        arr = data["events"]
        # 同一个任务从失败恢复为成功时，应更新事件，不能永久停留在 FAILED。
        previous=[x for x in arr if x.get("job_id")==event["job_id"]]
        if not previous or previous[-1].get("status") != event.get("status"):
            arr.append(event)
        data["events"] = arr[-200:]
        try:
            gh_write_json(path,data,sha)
            return
        except RuntimeError as err:
            if "HTTP 409" not in str(err) and "HTTP 422" not in str(err):
                raise
            time.sleep(1+attempt)
    raise RuntimeError("通知事件更新冲突，请人工检查 GitHub queue/events.json")

def list_pending_jobs():
    r = gh_read("jobs/pending",missing_ok=True)
    if r is None:
        return []
    if not isinstance(r,list):
        raise RuntimeError("jobs/pending 并非目录")
    pending = []
    for item in r:
        if item.get("type")=="file" and item["name"].endswith(".json"):
            txt,_=gh_read_text(item["path"])
            pending.append(json.loads(txt))
    return sorted(pending,key=lambda j:(j.get("created_at",""),j.get("id","")))

def validate_job(job):
    job_id = str(job.get("id",""))
    if not re.fullmatch(r"[a-zA-Z0-9_-]{8,64}", job_id):
        raise ValueError("任务 ID 不合法")
    if job.get("schema_version") != 1 or job.get("kind")!="image_edit":
        raise ValueError("任务 schema/kind 不支持")
    src = job.get("input_b64_path","")
    if src not in (f"jobs/inputs/{job_id}.jpg.b64",
                   f"jobs/inputs/{job_id}.jpeg.b64",
                   f"jobs/inputs/{job_id}.png.b64",
                   f"jobs/inputs/{job_id}.webp.b64"):
        raise ValueError("母图路径与 job ID 不匹配")
    sha = str(job.get("input_sha256",""))
    if not re.fullmatch(r"[a-f0-9]{64}",sha):
        raise ValueError("缺少母图 sha256")
    prompt = str(job.get("prompt","")).strip()
    if not 3 <= len(prompt) <= 2500:
        raise ValueError("编辑提示词为空或过长")
    resolution = int(job.get("resolution",512))
    if resolution not in (512,640,768):
        raise ValueError("只支持分辨率 512/640/768")
    seed = int(job.get("seed",42))
    if not 0 <= seed < 2**64:
        raise ValueError("seed 不合法")
    return job_id,src,sha,prompt,resolution,seed

def prepare_image(job_id, source_path, expected_sha, resolution):
    b64_txt,_ = gh_read_text(source_path)
    raw = base64.b64decode("".join(b64_txt.split()),validate=True)
    if len(raw)>MAX_SOURCE_BYTES:
        raise ValueError("母图压缩体积超过 4MB")
    actual_sha = hashlib.sha256(raw).hexdigest()
    if actual_sha != expected_sha:
        raise ValueError("母图 SHA256 不匹配")
    Image.MAX_IMAGE_PIXELS = 48_000_000
    image = ImageOps.exif_transpose(Image.open(io.BytesIO(raw))).convert("RGB")
    if image.width*image.height>48_000_000:
        raise ValueError("原图像素过大")
    cap=resolution**2
    if image.width*image.height>cap:
        ratio=(cap/(image.width*image.height))**0.5
        w=max(32,round(image.width*ratio/32)*32)
        h=max(32,round(image.height*ratio/32)*32)
        image=image.resize((w,h), Image.Resampling.LANCZOS)
    name=f"queue_{job_id}.png"
    (COMFY/"input").mkdir(exist_ok=True,parents=True)
    image.save(COMFY/"input"/name)
    print("母图:",job_id,"实际编辑尺寸:",image.size)
    return name

def generate_one(job_id, prompt, source_name, seed, resolution):
    global EDIT_RESOLUTION
    EDIT_RESOLUTION=resolution
    graph=make_graph(prompt,source_name,seed)
    graph["10"]["inputs"]["filename_prefix"]=f"Qwen21_{job_id}"
    assert graph["5"]["inputs"]["images.image_1"]==["4",0], "未连接参考图"
    resp=requests.post(URL+"/prompt",
                       json={"prompt":graph,"client_id":uuid.uuid4().hex},timeout=75)
    if not resp.ok:
        raise RuntimeError(f"ComfyUI 工作流校验失败: {resp.text[:950]}")
    body=resp.json()
    if body.get("node_errors"):
        raise RuntimeError("ComfyUI 节点错误："+str(body["node_errors"])[:800])
    prompt_id=body["prompt_id"]
    deadline=time.monotonic()+JOB_TIMEOUT_SECONDS
    while time.monotonic()<deadline:
        history=requests.get(URL+"/history/"+prompt_id,timeout=30).json()
        entry=history.get(prompt_id)
        if entry:
            status=entry.get("status",{})
            if not status.get("completed",False):
                errors=[m[1] for m in status.get("messages",[])
                        if isinstance(m,(list,tuple)) and len(m)==2 and m[0]=="execution_error"]
                if errors:
                    e=errors[-1]
                    raise RuntimeError(
                        "ComfyUI 节点异常 "
                        +str(e.get("node_id","?"))+"("+str(e.get("node_type","?"))+"): "
                        +str(e.get("exception_type","?"))+": "
                        +str(e.get("exception_message","?"))[:850]
                    )
                raise RuntimeError("ComfyUI 编辑失败："+str(status)[:750])
            images=entry.get("outputs",{}).get("10",{}).get("images",[])
            if not images:
                raise RuntimeError("任务结束但 SaveImage 无输出")
            info=images[0]
            path=OUT/info.get("subfolder","")/info["filename"]
            if not path.is_file():
                raise FileNotFoundError(str(path))
            return path
        time.sleep(3)
    raise TimeoutError("生成超时，任务仍可能在 ComfyUI 中执行，请检查当前运行时")

def execute_one(job):
    job_id,src,sha,prompt,res,seed=validate_job(job)
    result_path=f"jobs/results/{job_id}.json"
    previous=gh_read(result_path,missing_ok=True)
    if previous:
        previous_data,_=gh_read_text(result_path)
        if json.loads(previous_data).get("status")=="SUCCEEDED":
            print(f"⏭ 已成功，跳过: {job_id}")
            return "SKIPPED"
        print("发现旧的失败记录，允许继续重试:",job_id)
    image_path=f"images/{job_id}.png"
    # 在图片已提交、结果尚未写入的中断场景中复用原图（幂等）
    if not gh_read(image_path,missing_ok=True):
        source_name=prepare_image(job_id,src,sha,res)
        output_file=generate_one(job_id,prompt,source_name,seed,res)
        with Image.open(output_file) as output_image:
            output_image.verify()
        gh_write_bytes(image_path,output_file.read_bytes(),f"images: Qwen edit {job_id}")
        display(Image.open(output_file))
    image_url=f"https://github.com/{REPO}/blob/{BRANCH}/{image_path}"
    result={"schema_version":1,"job_id":job_id,"status":"SUCCEEDED",
            "input_path":src,"image_url":image_url,"finished_at":now_iso(),
            "seed":seed,"resolution":res}
    old_result=gh_read(result_path,missing_ok=True)
    gh_write_json(result_path,result,existing_sha=old_result["sha"] if old_result else None)
    event={"job_id":job_id,"status":"SUCCEEDED",
           "image_url":image_url,"finished_at":result["finished_at"]}
    append_event(event)
    print("✅ 成功：",image_url)
    return "SUCCEEDED"

def execute_pending(max_jobs=MAX_JOBS_PER_RUN):
    if not healthy():
        raise RuntimeError("ComfyUI 未运行，请先运行前面的启动单元")
    tasks=list_pending_jobs()
    print("GitHub 队列任务总数:",len(tasks))
    ran=0
    for job in tasks:
        if ran>=max_jobs: break
        job_id=str(job.get("id","UNKNOWN"))
        existing=gh_read(f"jobs/results/{job_id}.json",missing_ok=True)
        if existing:
            current_record,_=gh_read_text(f"jobs/results/{job_id}.json")
            if json.loads(current_record).get("status")=="SUCCEEDED":
                continue
        ran+=1
        try:
            print(f"\n▶️ 执行任务 {job_id} ({ran}/{max_jobs})")
            execute_one(job)
        except KeyboardInterrupt:
            print("执行已人工中断，未标记为失败，可重新启动。")
            raise
        except Exception as exc:
            brief=f"{type(exc).__name__}: {str(exc)[:800]}"
            brief=brief.replace(GITHUB_TOKEN,"[REDACTED]")
            print("❌ 本次运行失败（任务仍保留，修复后可重试）:",job_id,brief)
            if re.fullmatch(r"[A-Za-z0-9_-]{8,64}",job_id):
                # GPU/模型/网络问题是可重试故障，不将 pending 错误地标成永久 FAILED。
                attempt={"schema_version":1,"job_id":job_id,"status":"RETRYABLE_FAILED",
                         "error":brief,"last_attempt_at":now_iso()}
                try:
                    path=f"jobs/attempts/{job_id}.json"
                    existing_attempt=gh_read(path,missing_ok=True)
                    gh_write_json(path,attempt,existing_sha=existing_attempt["sha"] if existing_attempt else None)
                    print("📝 已记录可重试错误:",f"https://github.com/{REPO}/blob/{BRANCH}/{path}")
                except Exception as log_exc:
                    safe=str(log_exc).replace(GITHUB_TOKEN,"[REDACTED]")
                    print("⛔ GitHub 错误记录上传失败：",type(log_exc).__name__,safe[:650])
                    print("如果是 HTTP 403，请确认 Colab Secrets 的 Token 具备 Contents: Read and write。")
    if ran==0:
        print("当前没有新任务。你发来母图和提示词，我提交任务后再运行本单元即可。")
    else:
        print(f"本次处理 {ran} 个任务；结果在 GitHub jobs/results 和 images 中。")
    print("本单元执行完成，未启用后台轮询。")

execute_pending()


## 查看成果与失败记录

- [生成图片目录](https://github.com/mjw3177109/qwen2.1tu/tree/main/images)
- [任务结果目录](https://github.com/mjw3177109/qwen2.1tu/tree/main/jobs/results)
- [最新通知事件](https://github.com/mjw3177109/qwen2.1tu/blob/main/queue/events.json)

如果遇到显存、依赖或节点错误，在 ChatGPT 发对应结果 JSON / 错误日志即可定位。当前模型执行仍待真实 GPU 验收。